# Day 22: Self-Attention & The Transformer Revolution

> **Goal:** Understand the self-attention mechanism from first principles, implement it from scratch, and see how it differs from convolutions.

---

## The Big Idea

Convolutions process **local neighbourhoods**. But what if you need to relate distant parts of an image (or sentence)?

**Self-attention** allows every element to attend to (look at) every other element directly. No locality constraint!

### Analogy for optics
Think of convolution as a **local filter** (like a PSF with finite support). Self-attention is like a **global, content-dependent filter** — each pixel decides which other pixels are relevant based on their content.

## The "Attention Is All You Need" Paper (2017)

The Transformer architecture replaced recurrence with self-attention:
- Originally for NLP (machine translation)
- Now used in vision (ViT), audio, protein folding, etc.
- Foundation of GPT, BERT, and all modern LLMs

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np
import math

torch.manual_seed(42)

---
## 1. Intuition: Queries, Keys, and Values

Self-attention uses three projections of the input:

| Symbol | Name | Analogy |
|---|---|---|
| **Q** | Query | "What am I looking for?" |
| **K** | Key | "What do I contain?" |
| **V** | Value | "What information should I give?" |

### The attention formula:

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right) V$$

1. **QK^T** — compute similarity between all pairs of tokens
2. **/ √d_k** — scale to prevent softmax saturation
3. **softmax** — convert to attention weights (probabilities)
4. **× V** — weighted combination of values

In [ ]:
def scaled_dot_product_attention(Q, K, V):
    """
    Compute scaled dot-product attention from scratch.

    Args:
        Q: (batch, seq_len, d_k)
        K: (batch, seq_len, d_k)
        V: (batch, seq_len, d_v)

    Returns:
        output: (batch, seq_len, d_v)
        attention_weights: (batch, seq_len, seq_len)
    """
    d_k = Q.size(-1)

    # Step 1: Compute attention scores
    scores = torch.matmul(Q, K.transpose(-2, -1))  # (B, seq, seq)

    # Step 2: Scale
    scores = scores / math.sqrt(d_k)

    # Step 3: Softmax → attention weights
    attn_weights = F.softmax(scores, dim=-1)  # (B, seq, seq)

    # Step 4: Weighted sum of values
    output = torch.matmul(attn_weights, V)  # (B, seq, d_v)

    return output, attn_weights


# Example
batch_size = 1
seq_len = 4
d_k = 8

Q = torch.randn(batch_size, seq_len, d_k)
K = torch.randn(batch_size, seq_len, d_k)
V = torch.randn(batch_size, seq_len, d_k)

output, attn_weights = scaled_dot_product_attention(Q, K, V)

print(f"Q shape: {Q.shape}")
print(f"K shape: {K.shape}")
print(f"V shape: {V.shape}")
print(f"Output shape: {output.shape}")
print(f"Attention weights shape: {attn_weights.shape}")
print(f"\nAttention weights (each row sums to 1):")
print(attn_weights.squeeze().detach().numpy().round(3))
print(f"Row sums: {attn_weights.squeeze().sum(dim=-1).tolist()}")

### Visualizing the Attention Matrix

The attention weight matrix is `(seq_len × seq_len)`. Each row shows how much one token attends to every other token. Since we apply softmax, each row sums to 1 — it's a probability distribution over "where to look."

The heatmap below shows these weights: darker blue = higher attention between that query-key pair.

In [ ]:
# Visualize attention weights
fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(attn_weights.squeeze().detach().numpy(), cmap="Blues")
ax.set_xlabel("Key position")
ax.set_ylabel("Query position")
ax.set_title("Attention Weights")
plt.colorbar(im)

# Add text annotations
for i in range(seq_len):
    for j in range(seq_len):
        val = attn_weights[0, i, j].item()
        ax.text(j, i, f"{val:.2f}", ha="center", va="center", fontsize=10)

plt.tight_layout()
plt.show()

---
## 2. Self-Attention Module

In **self-attention**, Q, K, and V all come from the same input, transformed by learned linear projections.

In [ ]:
class SelfAttention(nn.Module):
    """Single-head self-attention from scratch."""

    def __init__(self, d_model):
        super().__init__()
        self.d_model = d_model

        # Learned projections
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)

    def forward(self, x):
        """
        x: (batch, seq_len, d_model)
        returns: (batch, seq_len, d_model), attention_weights
        """
        Q = self.W_q(x)  # (B, seq, d_model)
        K = self.W_k(x)
        V = self.W_v(x)

        output, attn_weights = scaled_dot_product_attention(Q, K, V)
        return output, attn_weights


# Test
sa = SelfAttention(d_model=16)
x = torch.randn(2, 6, 16)  # batch=2, seq_len=6, d_model=16
out, weights = sa(x)

print(f"Input:   {x.shape}")
print(f"Output:  {out.shape}")
print(f"Weights: {weights.shape}")

---
## 3. Multi-Head Attention

One attention head can only focus on one type of relationship. **Multi-head attention** runs several attention heads in parallel, each looking for different patterns.

$$\text{MultiHead}(Q, K, V) = \text{Concat}(\text{head}_1, ..., \text{head}_h) W^O$$

Each head operates on a lower-dimensional subspace: `d_k = d_model / num_heads`.

In [ ]:
class MultiHeadAttention(nn.Module):
    """Multi-head self-attention from scratch."""

    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"

        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads  # dimension per head

        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)  # output projection

    def forward(self, x):
        B, seq_len, _ = x.shape

        # Project
        Q = self.W_q(x)  # (B, seq, d_model)
        K = self.W_k(x)
        V = self.W_v(x)

        # Reshape to (B, num_heads, seq, d_k)
        Q = Q.view(B, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        K = K.view(B, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        V = V.view(B, seq_len, self.num_heads, self.d_k).transpose(1, 2)

        # Attention per head
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        attn_weights = F.softmax(scores, dim=-1)  # (B, heads, seq, seq)
        attn_output = torch.matmul(attn_weights, V)  # (B, heads, seq, d_k)

        # Concatenate heads
        attn_output = attn_output.transpose(1, 2).contiguous().view(B, seq_len, self.d_model)

        # Final projection
        output = self.W_o(attn_output)

        return output, attn_weights


# Test
mha = MultiHeadAttention(d_model=32, num_heads=4)
x = torch.randn(2, 8, 32)  # batch=2, seq=8, d_model=32
out, weights = mha(x)

print(f"Input:   {x.shape}")
print(f"Output:  {out.shape}")
print(f"Weights: {weights.shape}  (batch, heads, seq, seq)")

### Visualizing Multi-Head Attention Patterns

With 4 heads operating on `d_k = 8` dimensions each (instead of the full `d_model = 32`), each head is free to learn a **different attention pattern**. Some heads may attend to nearby tokens (local patterns), while others attend to distant tokens (global patterns). The plots below show the 4 heads' attention matrices side by side.

In [ ]:
# Visualize attention from different heads
fig, axes = plt.subplots(1, 4, figsize=(16, 3.5))
for h in range(4):
    im = axes[h].imshow(weights[0, h].detach().numpy(), cmap="Blues")
    axes[h].set_title(f"Head {h+1}")
    axes[h].set_xlabel("Key"); axes[h].set_ylabel("Query")
    plt.colorbar(im, ax=axes[h], fraction=0.046)

plt.suptitle("Attention Patterns from 4 Heads", fontsize=14)
plt.tight_layout()
plt.show()

---
## 4. Positional Encoding

Self-attention is **permutation-invariant** — it doesn't know the order of tokens! We must add **positional information**.

The original Transformer uses sinusoidal positional encoding:

$$PE_{(pos, 2i)} = \sin(pos / 10000^{2i/d})$$
$$PE_{(pos, 2i+1)} = \cos(pos / 10000^{2i/d})$$

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float()
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * -(math.log(10000.0) / d_model))

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        pe = pe.unsqueeze(0)  # (1, max_len, d_model)
        self.register_buffer("pe", pe)

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]


# Visualize
pe = PositionalEncoding(d_model=64)
encodings = pe.pe.squeeze().numpy()[:50, :]

plt.figure(figsize=(12, 4))
plt.imshow(encodings.T, aspect="auto", cmap="RdBu")
plt.xlabel("Position")
plt.ylabel("Dimension")
plt.title("Sinusoidal Positional Encoding")
plt.colorbar()
plt.tight_layout()
plt.show()

---
## 5. Comparing Attention vs Convolution

| Property | Convolution | Self-Attention |
|---|---|---|
| **Receptive field** | Local (kernel size) | Global (all positions) |
| **Weight sharing** | Same kernel everywhere | Content-dependent |
| **Parameters** | O(k² · C²) | O(d²) |
| **Computation** | O(k² · H · W) | O(N² · d) where N = H×W |
| **Inductive bias** | Translation equivariance | None (more flexible) |
| **Data needs** | Less (strong bias) | More (must learn from data) |

---
## 6. PyTorch's Built-in Multi-Head Attention

In [ ]:
# PyTorch provides nn.MultiheadAttention
mha_pytorch = nn.MultiheadAttention(embed_dim=32, num_heads=4, batch_first=True)

x = torch.randn(2, 8, 32)
output, weights = mha_pytorch(x, x, x)  # self-attention: Q=K=V=x

print(f"Input:   {x.shape}")
print(f"Output:  {output.shape}")
print(f"Weights: {weights.shape}")

---
## 🧪 Exercises

### Exercise 1: Attention for similar inputs
Create two sequences: one where tokens 0 and 5 are similar, and the rest are random. Compute attention and verify that token 0 attends to token 5.

### Exercise 2: Visualize scaling effect
Compute attention with and without the `√d_k` scaling. Plot the softmax distributions. What happens without scaling for large d_k?

### Exercise 3: Learned positional encoding
Implement learned positional encoding using `nn.Embedding(max_len, d_model)` instead of sinusoidal.

### Exercise 4: Cross-attention
Implement cross-attention where Q comes from one sequence and K, V come from another. When would you use this?

In [ ]:
# Exercises: Your code here

---
## 📝 Key Takeaways

| Concept | Summary |
|---|---|
| **Self-attention** | Every token attends to every other token |
| **Q, K, V** | Three learned projections of the input |
| **Scaled dot-product** | `softmax(QK^T / √d_k) · V` |
| **Multi-head** | Multiple attention heads capture different relationships |
| **Positional encoding** | Inject position info (sinusoidal or learned) |
| **Complexity** | O(N²) — expensive for long sequences |

**Tomorrow:** We'll build a complete Transformer Encoder block — combining attention with feed-forward networks, layer norm, and residual connections.